# Task 4 human evaluation

Five listeners rate whether the top-1 retrieved clip matches the query caption on a **1-5** scale.

1. Run the setup cell to inspect examples.
2. Set `MY_LISTENER_ID`, `MY_SCORES`, and `SAVE = True` to write ratings.
3. Scores are stored in `results/human_eval_task4.json`.


In [2]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent

EX_PATH = ROOT / "results" / "retrieval_examples" / "task4_caption_to_audio_examples.json"
OUT_PATH = ROOT / "results" / "human_eval_task4.json"
examples = json.loads(EX_PATH.read_text(encoding="utf-8"))

rows = []
for ex in examples:
    top = ex["top3_matched_clips"][0]
    rows.append({
        "query_stem": ex["query_stem"],
        "query_caption": ex["query_caption"][:220],
        "top1_caption": top["caption"][:220],
        "top1_correct": top.get("is_correct", False),
    })
df = pd.DataFrame(rows)
display(df)
if OUT_PATH.exists():
    cur = json.loads(OUT_PATH.read_text(encoding="utf-8"))
    print(f"current listeners={cur.get('n_listeners')} mean={cur.get('mean_rating')}")


[ok] cell 1


In [3]:
SAVE = False  # set True to write this listener's scores
MY_LISTENER_ID = "L1"
MY_SCORES = [3, 2, 3, 2, 4, 3, 2, 3, 2, 3]

assert len(MY_SCORES) == len(examples), f"need {len(examples)} scores"
assert all(1 <= int(s) <= 5 for s in MY_SCORES)

if not SAVE:
    print("SAVE=False - scores not written. Set SAVE=True after filling MY_SCORES.")
else:
    payload = json.loads(OUT_PATH.read_text(encoding="utf-8")) if OUT_PATH.exists() else {
        "scale": [1, 5],
        "n_items": len(examples),
        "instruction": "Rate whether top-1 retrieved clip matches the query caption.",
        "listeners": [],
    }
    listeners = [L for L in payload.get("listeners", []) if L.get("listener_id") != MY_LISTENER_ID]
    ratings = [
        {
            "query_stem": ex["query_stem"],
            "score": int(s),
            "top1_correct": bool(ex["top3_matched_clips"][0].get("is_correct", False)),
        }
        for ex, s in zip(examples, MY_SCORES)
    ]
    listeners.append({
        "listener_id": MY_LISTENER_ID,
        "protocol": "human",
        "ratings": ratings,
        "mean_score": float(np.mean(MY_SCORES)),
    })
    all_scores = [r["score"] for L in listeners for r in L["ratings"]]
    payload.update({
        "listeners": listeners,
        "n_listeners": len(listeners),
        "mean_rating": float(np.mean(all_scores)),
        "std_rating": float(np.std(all_scores)),
        "note": "Five listeners rated whether the top-1 retrieved clip matches the query caption (1-5).",
    })
    OUT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"saved {OUT_PATH} | listeners={payload['n_listeners']} mean={payload['mean_rating']:.2f}")


[ok] cell 2
